In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

from seqAE_model import SeqAutoencoder
from contra_seq_dataset import ContraSeqDataset, get_dataset_array, get_anc_map
from losses import SupConLoss, padce_loss
from tqdm.notebook import tqdm
import torch
import random

In [2]:
## Dataset

anc_path = 'model_dataset/anchor_smiles.csv'
aug_path = 'model_dataset/augmented_smiles_balanced.csv'

ds = ContraSeqDataset(anc_path, aug_path)
ds_arr = get_dataset_array(anc_path, aug_path)
anc_map = get_anc_map(ds_arr)

## Model instantiation

torch.cuda.empty_cache()
use_cuda = True
device =  torch.device("cuda" if use_cuda else "cpu")

supcon_loss = SupConLoss()
lr = 0.00001

In [3]:
## Loader
from contra_seq_dataset import AnchoredSampler
from torch.utils.data import DataLoader, RandomSampler

# # # # # # # # # # # # # # #
BATCH_SIZE = 16 # this is the number of ANCHORS
rand16 = random.sample(range(0,10000),32)

sampler = AnchoredSampler(sampler = RandomSampler(list(anc_map.keys())), 
                          anc_map = anc_map, batch_size = BATCH_SIZE, drop_last = True)

loader = DataLoader(ds, batch_sampler=sampler, num_workers=0, pin_memory=True)

In [4]:
def live_plot(data_dict, figsize=(10,5), title=f'Batch size: {BATCH_SIZE}'):
    clear_output(wait=True)
    plt.figure(figsize=figsize)
    for label,data in data_dict.items():
        plt.plot(data, label=label)
    plt.title(title)
    plt.grid(True)
    plt.xlabel('Run')
    plt.legend(loc='center left') # the plot evolves to the right
    plt.show();

In [5]:
from IPython.display import clear_output
import matplotlib.pylab as plt
import torch.nn as nn

torch.cuda.empty_cache()
use_cuda = True
device =  torch.device("cuda" if use_cuda else "cpu")

model = SeqAutoencoder(n_tokens = ds.n_tokens, max_len = 122,
                       dim_emb=512, heads=8, dim_hidden=32,
                       L_enc=6, L_dec=6, dim_ff=2048, 
                       drpt=0.1, actv='relu', eps=0.6, b_first=True)

if torch.cuda.device_count() > 1:
    print("Let's use", torch.cuda.device_count(), "GPUs!")
    model = nn.DataParallel(model)
    model.to(device)
else:
    model = model.to(device)
    
optimizer = torch.optim.Adam(model.parameters(), lr = lr)    
model.train()

run_data = {'Recon':[]}

for i in range(100):
    for samp in loader:
        
        optimizer.zero_grad()
        
        for k,v in samp.items():
            if torch.is_tensor(v):
                samp[k] = v.to(device)
                
        samp['out_mask'] = torch.zeros_like(samp['out_mask'])
        samp['pad_mask'] = torch.zeros_like(samp['pad_mask'])
        samp['avg_mask'] = torch.ones_like(samp['avg_mask'])
        
        _, dec_out = model.forward(samp['seq'], samp['pad_mask'], 
                                   samp['avg_mask'], samp['out_mask'])    

        recon_loss = padce_loss(samp['seq'], dec_out.squeeze(), 
                                samp['pad_mask'], samp['out_mask'])  

        run_data['Recon'].append(recon_loss.item())
        live_plot(run_data)

        loss = recon_loss
        loss.backward()
        optimizer.step()

KeyboardInterrupt: 

<Figure size 720x360 with 0 Axes>